In [ ]:
!pip install -q crewai crewai-tools
!pip install ddgs

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.9/238.9 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48

In [ ]:
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY2")

In [ ]:
from crewai import LLM

llm = LLM(
    model = "gemini/gemini-flash-lite-latest",
    temperature = 0.7,
)

In [ ]:
from crewai.tools import tool
from ddgs import DDGS

@tool("DuckDuckGo Search")
def search_tool(query: str) -> str:
  """Search the web using DuckDuckGo. Input is a search query.
  Returns the top results with title, link and a short snippet."""
  try:
    results = DDGS().text(query, max_results = 5)
    return "\n\n".join(
        f"Title: {r['title']}\nLink: {r['href']}\nSnippet: {r['body']}"
        for r in results
    )
  except Exception as e:
    return f"Search failed: {e}"

In [ ]:
from crewai_tools import ScrapeWebsiteTool, FileWriterTool

scrape_tool = ScrapeWebsiteTool()
file_writer = FileWriterTool()

In [ ]:
from crewai import Agent

site_scraper = Agent(
    role = "Website Scraper",
    goal = "Read competitor websites and extract what each product offers: "
           "main features, target customers and key selling points",
backstory = "You are an expert at reading websites and pulling out the"
            "important facts while ignoring menus , ads and filler text.",
tools = [scrape_tool],
llm = llm,
verbose = True,
allow_delegation = False,
)

pricing_analyst = Agent (
    role = "Pricing Analyst",
    goal = "Find how each quick-commerce app charges customers: delivery fees, "
         "handling/platform fees, minimum order value, surge charges and "
         "membership plans",
    backstory = "You track Indian quick-commerce pricing closely. You present "
              "fees clearly and never guess a number you could not find.",
    tools = [scrape_tool, search_tool],
    llm = llm,
    verbose = True,
    allow_delegation = False,

)

news_analyst = Agent(
    role = "News Analyst",
    goal = "Find the most important recent news about each competitor: "
         "launches, funding, partnerships and controversies",
    backstory = "You are a tech journalist who quickly spots which news "
              "actually matters for a company's strategy.",
    tools = [search_tool],
    llm = llm,
    verbose = True,
    allow_delegation = False,

)

strategist = Agent(
    role = "Strategist",
    goal = "Combine features, pricing and news into a clear SWOT analysis "
           "for each competitor and save it as a report",
    backstory = "You are a sharp business consultant who turns raw research "
                "into practical insights a founder can act on.",
    tools = [file_writer],
    llm = llm,
    verbose = True,
    allow_delegation = False,
)

In [ ]:
from crewai import Task

features_task = Task(
    description = "For each of these competitor websites: {competitors},"
                  "read the website and extract the main features, target"
                  "customers and key selling points.",
    expected_output = "A secrion per competitior listing its main features, "
                      "target customers and selling points.",
)

pricing_task = Task(
    description = "For each of these quick-commerce apps: {competitors}, find "
                "their delivery fees, handling/platform fees, minimum order "
                "value, surge or rain charges and membership plans. Use their "
                "websites and web search. If a number is not found, write "
                "'Not found' instead of guessing.",
    expected_output= "A fee and membership comparison table for each app, "
                    "with sources.",
)
news_task = Task(
    description="Search the web for the most important recent news about "
                "each of these competitors: {competitors}. Focus on launches, "
                "funding, partnerships and controversies.",
    expected_output="3-5 recent news points per competitor, each with a "
                    "source link.",

)

swot_task = Task(
    description = "Using the features, pricing and news research, write a SWOT"
                  "analysis (Strengths, Weaknesses, Opportunities, Threats) for "
                  "each competitor, followed by a short summary of key takeaways."
                  "Save the final report using the file writer tools with the "
                  "filename 'report.md'. ",
    expected_output= "A Markdown report with a SWOT table for each competitor"
                     "and a key takeaways section, saved as report.md.",
    context = [features_task, pricing_task, news_task],


)

In [ ]:
from crewai import Crew, Process

competitor_crew = Crew(
    agents = [site_scraper, pricing_analyst, news_analyst, strategist],
    tasks = [features_task, pricing_task, news_task, swot_task],
    process = Process.hierarchical,
    manager_llm = llm, max_rpm = 8,
    verbose = True,
)

result = await competitor_crew.kickoff_async(inputs = {
    "competitors" :"Zepto (https://www.zeptonow.com),"
                   "Swiggy Instamart (https://www.swiggy.com/instamart),"
                   "Blinkit (https://blinkit.com)"


})

╭───────────────────────── 🚀 Crew Execution Started ──────────────────────────╮
│                                                                              │
│  Crew Execution Started                                                      │
│  Name: crew                                                                  │
│  ID: b3a51cb6-3513-43c5-ade7-693e33ce747f                                    │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────── 📋 Task Started ───────────────────────────────╮
│                                                                              │
│  Task Started                                                                │
│  Name: For each of these competitor websites: Zepto                          │
│  (https://www.zeptonow.com)

In [ ]:
with open("report.md", "w") as f:
    f.write(result.raw)

print("File exists:", os.path.exists("report.md"))

with open("report.md") as f:
    display(Markdown(f.read().replace("$", r"\$")))

File exists: True


# Quick-Commerce Competitor Analysis: SWOT Report & Key Takeaways

## 1. Zepto

| Strengths | Weaknesses |
| :--- | :--- |
| • **Zero Fee Model:** Completely eliminated handling/platform fees and surge/rain charges across all orders, driving high consumer appeal.<br>• **Strong Financial Backing & Valuation:** Raised \$450M in pre-IPO funding at a \$7.0B valuation and confidentially filed for a \$1.2B IPO.<br>• **Hyper-Speed Logistics:** Efficient dark store network and infrastructure optimized for ultra-fast 10-minute grocery and daily essentials delivery.<br>• **Value-Added Offerings:** Introduction of "Zepto Cafe" to capture higher-margin quick-service food and beverage snacking occasions. | • **Narrower Ecosystem Synergy:** Operates primarily as a standalone quick-commerce app compared to Swiggy's integrated food and delivery ecosystem.<br>• **Labor & Regulatory Friction:** Faced delivery partner protests regarding payout structures, gig worker welfare, and delivery safety parameters.<br>• **Narrow Margin Pressures:** Zero handling and surge charges combined with heavy discounting may strain unit economics ahead of public listing. |

| Opportunities | Threats |
| :--- | :--- |
| • **IPO Capital Deployment:** Utilizing anticipated public listing funds to aggressively expand geographic footprint and dark store density.<br>• **Category Expansion:** Scaling non-grocery, electronics, and Zepto Cafe offerings to increase average order value (AOV) and customer lifetime value.<br>• **Loyalty Monetization:** Further optimizing the Zepto Pass subscription tier to secure repeat purchases and brand loyalty. | • **Intense Market Competition:** Fierce battle for market share against well-capitalized giants like Blinkit (Zomato) and Swiggy Instamart.<br>• **Gig Worker Unrest:** Potential for recurring labor strikes, unionization pressures, or regulatory crackdowns on gig work economics.<br>• **Operational Cost Inflation:** Rising real estate and warehousing costs in prime urban clusters. |

---

## 2. Swiggy Instamart

| Strengths | Weaknesses |
| :--- | :--- |
| • **Integrated Ecosystem & Cross-Selling:** Seamlessly integrated within the broader Swiggy app, allowing instant cross-pollination with food delivery users.<br>• **High-Value Loyalty Program:** Backed by Swiggy One and Swiggy One Lite, providing built-in retention through unified delivery benefits.<br>• **Established Brand & Logistics:** Backed by Swiggy's robust delivery network and successful public market backing (Swiggy IPO).<br>• **Strategic Brand Alliances:** Secured exclusive inventory partnerships with top national and international consumer goods brands. | • **Complex Fee Structure:** Applies variable handling fees (₹7–₹12 + GST) and dynamic surge/rain charges which can alienate price-sensitive users.<br>• **Execution Complexities:** Balancing multiple verticals (food delivery, dine-out, quick-commerce) can dilute hyper-focus compared to pure-play rivals.<br>• **Delivery Partner Discontent:** Experience with urban delivery partner strikes over revised distance calculation and payout methodologies. |

| Opportunities | Threats |
| :--- | :--- |
| • **Non-Grocery Scaling:** Accelerated expansion into high-margin categories like electronics, appliances, beauty products, and festive gifting.<br>• **Post-IPO War Chest:** Deploying fresh capital from Swiggy's public offering to aggressively scale dark stores and out-compete rivals.<br>• **Tier-2 & Tier-3 Expansion:** Replicating urban success in emerging semi-urban markets with tailored convenience offerings. | • **Aggressive Price Wars:** Margin erosion driven by competitors offering zero handling or surge fees.<br>• **Regulatory Scrutiny:** Increasing municipal and labor oversight regarding dark store zoning and delivery partner compensation.<br>• **Market Saturation:** Rapid expansion by Blinkit and Zepto threatening Swiggy’s existing quick-commerce market share. |

---

## 3. Blinkit (formerly Grofers)

| Strengths | Weaknesses |
| :--- | :--- |
| • **Pioneer Advantage & Scale:** First mover on the 10-minute delivery model in India with a rapidly expanding, highly optimized dark store network.<br>• **Financial Backing & Profitability Trajectory:** Parent company Zomato reported stellar progress on Blinkit's path to profitability, outperforming market expectations.<br>• **High-Value Unique Offerings:** Innovative service offerings such as on-demand document printing and high-profile electronics/Apple reseller partnerships.<br>• **Diverse Product Catalog:** Successfully expanded beyond groceries into electronics, stationery, beauty, and over-the-counter medicine. | • **High Friction Fees:** Imposes standard small-order fees, handling fees (₹9 to ₹25+), and dynamic surge/rain charges (up to ₹50) during peak hours.<br>• **Localized Zoning & Regulatory Pressures:** Faces municipal scrutiny regarding dark store congestion, residential zoning laws, and safety compliance.<br>• **Variable Membership Model:** Lacks a ubiquitous, standardized standalone loyalty subscription pass across all regional markets. |

| Opportunities | Threats |
| :--- | :--- |
| • **High-Margin Electronics & D2C Partnerships:** Deepening exclusive brand partnerships for lightning-fast delivery of high-ticket gadgets and electronics.<br>• **Dark Store Optimization:** Leveraging advanced mapping and predictive analytics to further reduce fulfillment times and operational overhead.<br>• **Service Diversification:** Expanding unique utility services (like printing and quick-gifting) to capture non-grocery consumer mindshare. | • **Intense Competitive Pressures:** Zepto’s zero-fee model and Swiggy’s ecosystem integration pose continuous threats to customer acquisition.<br>• **Municipal and Neighborhood Pushback:** Stricter local regulations on dark stores in residential areas could disrupt fulfillment infrastructure.<br>• **Customer Churn on Surge Pricing:** High surge fees and handling charges during peak hours may drive price-sensitive users to competitors. |

---

## Key Takeaways

1. **Intense Price and Fee Warfare:** Platforms are radically differentiating through their fee structures. While Zepto has captured consumer mindshare by eliminating handling fees and surge charges entirely, Swiggy Instamart and Blinkit rely on dynamic handling, surge, and small-order fees to manage unit economics, creating a constant tension between profitability and customer acquisition.
2. **Diversification Beyond Groceries:** The battleground has firmly shifted from basic daily essentials to high-margin, high-ticket categories. Blinkit’s partnerships with Apple and print services, Swiggy’s push into electronics and beauty, and Zepto’s launch of Zepto Cafe highlight a universal strategy to boost Average Order Value (AOV).
3. **Capital and Public Markets Maturity:** All three major players are aggressively funded or publicly backed (Zepto’s pre-IPO funding and confidential IPO filing, Swiggy’s successful IPO, and Zomato-backed Blinkit’s profitability milestones), signaling that quick-commerce has graduated from a cash-burning experiment to a core pillar of Indian retail.
4. **Shared Operational Bottlenecks:** Across the board, all competitors face identical systemic challenges: delivery partner pushback regarding gig payout structures, and municipal/zoning scrutiny over dark store congestion in dense urban neighborhoods.